In [7]:
import $ivy.`org.typelevel::cats-core:2.3.0`

import $ivy.$                               

In [8]:
import cats.implicits._, cats.syntax.all._, cats.data._

import cats.implicits._, cats.syntax.all._, cats.data._

In [9]:
type Word=String

defined type Word

In [10]:
val dictionary: Set[String] = Set("pussy", "phase", "plash", "happy")

dictionary: Set[String] = Set("pussy", "phase", "plash", "happy")

In [11]:
sealed trait Tag
case object Green extends Tag
case object Yellow extends Tag
case object Gray extends Tag
case object Red extends Tag

defined trait Tag
defined object Green
defined object Yellow
defined object Gray
defined object Red

In [12]:
type TaggedWord = List[(Char, Tag)]

defined type TaggedWord

In [14]:
def mapState[A, B, S](l: List[A])(f: A => State[S, B]): State[S, List[B]] =
    l.traverse(f)
    
implicit class ListOps[A](l: List[A]){
    def removeFirst(a: A): List[A] = 
        l match {
            case Nil => Nil
            case head :: tail if head == a => tail
            case head :: tail => head :: tail.removeFirst(a)
        }
}
    

defined function mapState
defined class ListOps

In [19]:
import $plugin.$ivy.`org.typelevel::kind-projector:0.10.3`

import $plugin.$                                          

In [20]:
List(1,2,3).traverse[State[String, ?], Int](i => State{ s: String => (s, i+1) }).runA("").value

res19: List[Int] = List(2, 3, 4)

In [26]:
case class Wordle(dictionary: Set[String], chosen: String, attempts: List[TaggedWord])

object Wordle{
    
    def init(dict: Set[String], chosen: String): Wordle = 
        Wordle(dict, chosen, List())
    
    def attempt(wordle: Wordle, word: String): Wordle = 
        wordle.copy(attempts = tag(word, wordle.dictionary, wordle.chosen) :: wordle.attempts)
    
    import scala.collection.mutable.ListBuffer
    
    def tag(word: String, dict: Set[String], chosen: String): TaggedWord = 
        if (!dict.contains(word)) word.map((_, Red)).toList
        else word.zip(chosen).toList.traverse{
            case (l1, l2) => State{ rest: List[Char] => 
                if (l1 == l2) (rest.removeFirst(l1), (l1, Green))
                else if (rest contains l1) (rest.removeFirst(l1), (l1, Yellow))
                else (rest, ((l1, Gray)))
            }
        }.runA(chosen.toList).value
}


defined class Wordle
defined object Wordle

In [27]:
List("happy", "pussy", "plash", "phase")
    .foldLeft(Wordle.init(dictionary, "phase"))(Wordle.attempt)

res26: Wordle = Wordle(
  Set("pussy", "phase", "plash", "happy"),
  "phase",
  List(
    List(('p', Green), ('h', Green), ('a', Green), ('s', Green), ('e', Green)),
    List(('p', Green), ('l', Gray), ('a', Green), ('s', Green), ('h', Yellow)),
    List(('p', Green), ('u', Gray), ('s', Yellow), ('s', Green), ('y', Gray)),
    List(('h', Yellow), ('a', Yellow), ('p', Yellow), ('p', Gray), ('y', Gray))
  )
)